## Problem: Implement Mixed Precision Training Using `torch.amp`

### Problem Statement
Mixed precision training uses both 16-bit and 32-bit floating-point types to accelerate training and reduce memory usage without significantly affecting model performance. Your task is to implement mixed precision training for a deep learning model using PyTorch's `torch.amp`.

### Requirements

1. **Enable Mixed Precision Training**:
   - Context manager to enable mixed precision for the forward pass.
   - Scale gradients during backpropagation and ensure stability.

In [7]:
# Implement mixed precision training in PyTorch using torch.amp
import torch.amp as amp
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import torch

In [8]:
# Generate synthetic data
X = torch.randn(1000, 10)
y = torch.randn(1000, 1)
dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [9]:
# Define a simple model
class SimpleModel(nn.Module):
    def __init__(self):
        super(SimpleModel, self).__init__()
        self.fc = nn.Linear(10, 1)

    def forward(self, x):
        return self.fc(x)

In [10]:
# Initialize model, loss function, and optimizer
model = SimpleModel().cuda()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Enable mixed precision training
scaler = amp.grad_scaler.GradScaler("cuda")

In [11]:
# Training loop
epochs = 5
for epoch in range(epochs):
    for inputs, labels in dataloader:
        inputs, labels = inputs.cuda(), labels.cuda()

        # Forward pass under autocast
        with amp.autocast("cuda"):
            outputs = model(inputs)
            loss = criterion(outputs, labels)

        # Backward pass with scaled gradients
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

    print(f"Epoch {epoch + 1}/{epochs}, Loss: {loss.item():.4f}")

Epoch 1/5, Loss: 1.9336
Epoch 2/5, Loss: 2.3950
Epoch 3/5, Loss: 0.7702
Epoch 4/5, Loss: 1.0352
Epoch 5/5, Loss: 1.6230


In [12]:
# Test the model on new data
X_test = torch.randn(5, 10).cuda()
with torch.no_grad(), torch.amp.autocast("cuda"):
    predictions = model(X_test)
    print("Predictions:", predictions)

Predictions: tensor([[-0.0309],
        [ 0.6973],
        [ 0.2108],
        [ 0.2825],
        [ 0.0252]], device='cuda:0', dtype=torch.float16)
